# 데이터 탐색 노트북

AI Hub에서 받은 데이터의 구조를 파악하고 클래스별 이미지 수를 확인합니다.

**먼저 이 노트북을 돌려서 데이터 구조 확인 후 config.yaml의 target_classes를 조정하세요.**

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

from src.config import load_config

config = load_config('../config.yaml')
raw_dir = Path('..') / config['data']['raw_dir']
print(f'데이터 경로: {raw_dir.resolve()}')
print(f'존재 여부: {raw_dir.exists()}')

## 1. 폴더 구조 확인

In [ ]:
# 최상위 폴더 목록 (클래스가 되어야 함)
class_dirs = sorted([d for d in raw_dir.iterdir() if d.is_dir()])
print(f'총 클래스 수: {len(class_dirs)}')
print('\n첫 20개:')
for d in class_dirs[:20]:
    print(f'  - {d.name}')

## 2. 클래스별 이미지 수

In [ ]:
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
class_counts = {}
for d in class_dirs:
    images = [p for p in d.rglob('*') if p.suffix.lower() in image_extensions]
    class_counts[d.name] = len(images)

sorted_counts = sorted(class_counts.items(), key=lambda x: -x[1])

print('이미지 많은 순 상위 20:')
for name, count in sorted_counts[:20]:
    print(f'  {name}: {count}')

print('\n이미지 적은 순 하위 10:')
for name, count in sorted_counts[-10:]:
    print(f'  {name}: {count}')

In [ ]:
# 분포 시각화
counts = list(class_counts.values())
plt.figure(figsize=(10, 4))
plt.hist(counts, bins=30)
plt.xlabel('Images per class')
plt.ylabel('Number of classes')
plt.title('Class size distribution')
plt.show()

print(f'평균: {sum(counts)/len(counts):.0f}')
print(f'최소: {min(counts)}')
print(f'최대: {max(counts)}')
print(f'총 이미지 수: {sum(counts)}')

## 3. 샘플 이미지 시각화

In [ ]:
from PIL import Image
import random

# 처음 8개 클래스에서 샘플 1장씩
sample_classes = class_dirs[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, class_dir in zip(axes.flat, sample_classes):
    images = [p for p in class_dir.rglob('*') if p.suffix.lower() in image_extensions]
    if images:
        img = Image.open(random.choice(images))
        ax.imshow(img)
        ax.set_title(class_dir.name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. 추천 클래스 목록 생성

이미지 수 기준으로 상위 N개 클래스를 config.yaml에 복사하기 좋은 형식으로 출력.

In [ ]:
N = 10  # 처음엔 10개로 시작 권장
top_classes = [name for name, _ in sorted_counts[:N]]

print(f'config.yaml의 target_classes에 복사:\n')
print('target_classes:')
for name in top_classes:
    print(f'  - "{name}"')